# Pipeline reproducible de reservas

Este notebook documenta la línea base pandas equivalente a `pipeline.py`. Lee las tablas de `reservas_db` en CockroachDB E3 y aplica el mismo filtro, join, calendario, agregación por período y clasificación por tamaño.

Requisitos: `pandas`, `sqlalchemy`, `psycopg` y un motor Parquet como `pyarrow`. La variable `RESERVAS_PANDAS_URL` permite sustituir la conexión predeterminada.

In [ ]:
from baseline import (
    PandasDbConfig,
    agregar_dimensiones_temporales,
    agregar_participantes_por_periodo,
    cargar_fuentes,
    categorizar_numero_participantes,
    crear_motor,
    exportar_parquet,
    filtrar_reservas_activas,
    unir_solicitudes_con_reservas,
)

config = PandasDbConfig.desde_entorno()
motor = crear_motor(config)

## 1. Lectura

Las fuentes corresponden exactamente a las tablas declaradas en `db/schema.sql`.

In [ ]:
fuentes = cargar_fuentes(motor)

## 2. Filtrado y join

Se conservan reservas programadas o en curso y se relacionan 1:1 con su solicitud.

In [ ]:
reservas_activas = filtrar_reservas_activas(fuentes["reservas"])
datos = unir_solicitudes_con_reservas(
    fuentes["solicitudes_reserva"],
    reservas_activas,
)

## 3. Fechas, agregación y clasificación

Se derivan año, trimestre y mes; luego se replica la suma Window por laboratorio y trimestre. Finalmente, el número de participantes se asigna a los mismos intervalos del `Bucketizer` de Spark.

In [ ]:
datos = agregar_dimensiones_temporales(datos)
datos = agregar_participantes_por_periodo(datos)
resultado = categorizar_numero_participantes(datos)
resultado.head()

## 4. Exportación

El resultado reproducible se almacena como `spark/out/reservas_procesadas.parquet`.

In [ ]:
destino = exportar_parquet(resultado)
motor.dispose()
destino